# Reed-Solomon Decoder

In [10]:
%load_ext autoreload
%autoreload 1
%aimport classes.galois_field
%aimport classes.galois_poly

from classes.galois_field import *
from classes.galois_poly import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
def do_unpack(value, degree, gf):
    mask    = gf._order
    ak      = [(value >> i*gf._field_m) & mask for i in range(degree+1)]
    ak      = [gf._get_power(a) for a in ak]
    return GaloisPoly(gf, ak[::-1])

## Get Syndromes

In [12]:
gf_extended = GaloisField(4, 0b10011)
root        = [GFE1, 1, 2, 3, 4, 5]

r   = 0x56D09_F171B_1E06F
r   = do_unpack(r, 14, gf_extended)

syn = [r(a) for a in root]

syn


Field Closed Succesfully!, 15 Non-Zero Elements


[1, 14, 0, 10, 9, 13]

In [13]:
def get_discrepancy(mu, syn, Lx):
    d_mu    = syn[mu]
    for i in range(1, Lx._degree+1): 
        # passes through all coeffs of localizator polynomial 
        term    = gf_extended.mul(Lx @ i, syn[mu - i])[ID_POWER]
        # Lx @ i take coefficient of degree "i" from the locator polynomial
        # syn[mu - i]
        d_mu    = gf_extended.add(d_mu, term)[ID_POWER]
        # discrpancy in step i
    return d_mu

In [14]:
# berlekamp-massey by steps
x   = do_unpack(0x10, 1, gf_extended)

lm1 = do_unpack(0x1, 0, gf_extended)
dm1 = GFE1

l0  = do_unpack(0x1, 0, gf_extended)
d0  = syn[0]

dx  = x*gf_extended.div(d0, dm1)[ID_POWER]
l1  = l0 + dx*lm1
d1  = get_discrepancy(1, syn, l1)

dx, l1, d1
print(l1, "\n")

dx  = x*gf_extended.div(d1, d0)[ID_POWER]
l2  = l1 + dx*l0
d2  = get_discrepancy(2, syn, l2)

dx, l2, d2
print(l2, "\n")

dx  = (x*x)*gf_extended.div(d2, d0)[ID_POWER]
l3  = l2 + dx*l0
d3  = get_discrepancy(3, syn, l3)

dx, l3, d3
print(l3, "\n")

l4  = l3
d4  = get_discrepancy(4, syn, l4)

l4, d4
print(l4, "\n")

dx  = (x*x)*gf_extended.div(d4, d2)[ID_POWER]
l5  = l4 + dx*l2
d5  = get_discrepancy(5, syn, l5)

dx, l5, d5
print(l5, "\n")

dx  = (x)*gf_extended.div(d5, d4)[ID_POWER]
l6  = l5 + dx*l4

dx, l6
print(l6, "\n")


a^1*x + 1 

a^13*x + 1 

a^10*x^2 + a^13*x + 1 

a^10*x^2 + a^13*x + 1 

a^5*x^3 + a^6*x^2 + a^13*x + 1 

a^4*x^3 + a^1*x^2 + 1 



In [15]:
print(do_unpack(0x10, 1, gf_extended))

x + 


## Chiens evaluation

In [16]:
for k in range(l6._degree + 1):
    print(k, l6 @ k)
    

0 0
1 -1
2 1
3 4


In [ ]:
# (ascending power: a0, a1, ..., an)
for k in range(l6._degree + 1):
    alpha_k_idx = l6 @ k                             # alpha power of coeff of x^k (GFE0 = -1 -> zero)
    value, pw   = gf_extended(alpha_k_idx)           # (binary value, alpha power)
    alpha       = "0" if alpha_k_idx == GFE0 else f"a^{pw}"
    print(f"x^{k}: {alpha:5} = {value:0{gf_extended._field_m}b}")

x^0: a^0   = 0001
x^1: 0     = 0000
x^2: a^1   = 0010
x^3: a^4   = 0011


In [ ]:
# Chien search: error at position i if L(alpha^-i) == 0
#   alpha^(-i) = alpha^(15-i)  ->  gf.inv(i)[ID_POWER]
def _chien_search(Lx, gf):
    err_pos = []
    for i in range(gf._order):
        a_inv = gf.inv(i)[ID_POWER]         # alpha^-i (power index)
        if Lx(a_inv) == GFE0:               # root of the locator polynomial
            err_pos.append(i)
    if len(err_pos) != Lx._degree:
        print(f"Warning: {len(err_pos)} roots found, expected {Lx._degree} (uncorrectable)")
    return err_pos

err_pos = _chien_search(l6, gf_extended)
err_pos